# 🎙 ePub → Audiobook M4B avec F5-TTS Français (GPU Colab)

Convertit un ePub en audiobook M4B avec chapitres marqués, en utilisant **F5-TTS RASPIAUDIO French** sur **GPU T4 gratuit**.

## 📋 Avant de commencer

1. **Active le GPU** : `Exécution` → `Modifier le type d'exécution` → `T4 GPU` → `Enregistrer`
2. Exécute les cellules dans l'ordre (Shift+Entrée sur chacune)
3. Quand on te demande un fichier, utilise le menu **Fichiers** à gauche pour uploader

## ⏱ Temps estimé

- Setup initial : ~5 min
- Génération livre Drizzt (~30 ch., ~8h audio final) : **~30-60 min sur GPU T4**
- Total : ~1h tranquille

## 1️⃣ Vérification GPU + installation des packages

In [ ]:
!nvidia-smi | head -10
print('---')
!pip install -q f5-tts soundfile pydub ebooklib beautifulsoup4 2>&1 | tail -5
!apt-get -qq install -y ffmpeg
print('✓ Setup terminé')

## 2️⃣ Téléchargement du modèle F5-TTS Français RASPIAUDIO (1.35 Go)

Le modèle est fine-tuné sur **120k samples LibriVox français**, multi-speakers.

In [ ]:
import os, urllib.request
os.makedirs('models', exist_ok=True)

files = {
    'models/model_last_reduced.pt': 'https://huggingface.co/RASPIAUDIO/F5-French-MixedSpeakers-reduced/resolve/main/model_last_reduced.pt?download=true',
    'models/vocab.txt': 'https://huggingface.co/RASPIAUDIO/F5-French-MixedSpeakers-reduced/resolve/main/vocab.txt?download=true',
}
for path, url in files.items():
    if not os.path.exists(path):
        print(f'Téléchargement {path}…')
        urllib.request.urlretrieve(url, path)
    sz = os.path.getsize(path)/1024/1024
    print(f'  ✓ {path} ({sz:.1f} Mo)')
print('Done.')

## 3️⃣ Upload de ton ePub

Exécute la cellule, puis sélectionne ton fichier ePub via le bouton qui apparaît.

In [ ]:
from google.colab import files
print('Sélectionne ton ePub :')
uploaded = files.upload()
epub_path = list(uploaded.keys())[0]
print(f'✓ {epub_path} ({len(uploaded[epub_path])/1024/1024:.1f} Mo)')

## 4️⃣ Voix de référence pour le clonage

F5-TTS clone une voix à partir d'un échantillon audio de **5-15 secondes** + sa transcription exacte.

**Option A — Sample fourni par défaut** : un narrateur LibriVox français téléchargé automatiquement (voix homme posée).

**Option B — Ton propre sample** : upload un WAV de 5-15s + tape la transcription dans la cellule suivante.

In [ ]:
# Option A : sample par défaut depuis le repo officiel F5-TTS (mâle anglais)
# Le modèle FR-tuné RASPIAUDIO applique l'accent français de toute façon
REF_AUDIO = 'ref_default.wav'
REF_TEXT = "Some call me nature, others call me mother nature."

if not os.path.exists(REF_AUDIO):
    print('Téléchargement sample de référence officiel F5-TTS…')
    url = 'https://raw.githubusercontent.com/SWivid/F5-TTS/main/src/f5_tts/infer/examples/basic/basic_ref_en.wav'
    try:
        urllib.request.urlretrieve(url, REF_AUDIO)
    except Exception as e:
        print(f'✗ Echec download : {e}')
        REF_AUDIO = None

# VALIDATION CRITIQUE : le sample DOIT exister pour la suite
assert REF_AUDIO is not None and os.path.exists(REF_AUDIO), (
    "❌ REF_AUDIO manquant ! Utilise l'Option B ci-dessous pour uploader ton propre WAV."
)

sz_kb = os.path.getsize(REF_AUDIO)/1024
print(f'✓ {REF_AUDIO} ({sz_kb:.0f} Ko)')
print(f'  Transcript : "{REF_TEXT}"')
from IPython.display import Audio
display(Audio(REF_AUDIO))


In [ ]:
# Option B : décommente si tu veux uploader TON sample
# from google.colab import files
# uploaded = files.upload()  # upload ton .wav
# REF_AUDIO = list(uploaded.keys())[0]
# REF_TEXT = "Tape ici la transcription EXACTE du WAV uploadé."
# print(f'✓ {REF_AUDIO}')
pass

## 5️⃣ Extraction des chapitres de l'ePub

Parse l'ePub, extrait les chapitres, te montre la liste pour validation.

In [ ]:
import zipfile, re
from bs4 import BeautifulSoup

def extract_chapters(epub_path):
    chapters = []
    with zipfile.ZipFile(epub_path) as z:
        html_files = sorted([n for n in z.namelist() if n.lower().endswith(('.html', '.xhtml'))])
        for fn in html_files:
            try:
                raw = z.read(fn).decode('utf-8', errors='replace')
            except: continue
            soup = BeautifulSoup(raw, 'html.parser')
            # Titre = h1/h2 du chapitre OU nom de fichier
            title_el = soup.find(['h1', 'h2', 'h3'])
            title = title_el.get_text(strip=True) if title_el else os.path.basename(fn)
            # Texte complet
            text = soup.get_text(' ', strip=True)
            text = re.sub(r'\s+', ' ', text).strip()
            # Filtre : skip si trop court (page de garde, copyright, etc.)
            if len(text) < 300:
                continue
            chapters.append({'title': title[:120], 'text': text, 'file': fn})
    return chapters

chapters = extract_chapters(epub_path)
print(f'✓ {len(chapters)} chapitres détectés :\n')
for i, ch in enumerate(chapters):
    print(f'  {i+1:2}. {ch["title"]:<60}  ({len(ch["text"]):,} chars)')
print(f'\nTotal : {sum(len(c["text"]) for c in chapters):,} chars')

## 6️⃣ Sélection des chapitres à convertir

Par défaut : **tous**. Modifie `selected_ids` si tu veux n'en convertir que certains (numéros = ceux affichés ci-dessus).

In [ ]:
# Pour ne convertir QUE certains chapitres, mets leurs numéros ici (ex: [3, 5, 7])
# Pour TOUS : laisse vide []
selected_ids = []

if not selected_ids:
    selected_chapters = chapters
else:
    selected_chapters = [chapters[i-1] for i in selected_ids]
print(f'✓ {len(selected_chapters)} chapitres à convertir')
total_chars = sum(len(c['text']) for c in selected_chapters)
print(f'  {total_chars:,} chars total')
print(f'  Estimation audio : ~{total_chars*0.06/60:.0f} minutes')

## 7️⃣ Chargement de F5-TTS sur GPU

Le chargement initial prend ~30s.

In [ ]:
import time
from f5_tts.api import F5TTS

print('Chargement F5-TTS sur GPU…')
t0 = time.time()
f5 = F5TTS(
    model='F5TTS_Base',
    ckpt_file='models/model_last_reduced.pt',
    vocab_file='models/vocab.txt',
    device='cuda',
)
print(f'✓ F5-TTS prêt en {time.time()-t0:.0f}s sur GPU')

## 8️⃣ Génération audio chapitre par chapitre

C'est le gros morceau. Estimation : ~1 min par minute d'audio final sur T4.

In [ ]:
import re, soundfile as sf, numpy as np
from tqdm import tqdm

# === VALIDATION CRITIQUE avant de lancer ===
assert REF_AUDIO and os.path.exists(REF_AUDIO), (
    f"❌ REF_AUDIO invalide : {REF_AUDIO!r}. Re-exécute la cellule 4."
)
assert REF_TEXT and isinstance(REF_TEXT, str), "❌ REF_TEXT manquant"
print(f'✓ ref_audio : {REF_AUDIO} ({os.path.getsize(REF_AUDIO)/1024:.0f} Ko)')
print(f'✓ ref_text  : "{REF_TEXT[:80]}..."')

# Liaisons FR forcées (trick du trait d'union — même règles que kokoro_server.py)
V = r'aeiouéèêëàâîïôöùûüœhAEIOUÉÈÊËÀÂÎÏÔÖÙÛÜŒH'
LIAISONS = [
    (re.compile(rf'\b(les|des|mes|tes|ses|ces|nos|vos|leurs|aux|quelques|plusieurs)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(nous|vous|ils|elles|on)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(en|y)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(un|aucun|mon|ton|bon)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(grand|petit|gros|haut|tout|tous|vingt|cent|premier|dernier)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(chez|sous|sans|dans|après)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(est|sont|était|peut|veut|fait|sait|dit)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
    (re.compile(rf'\b(mais|puis|donc|quand|plus|moins)\s+([{V}])', re.IGNORECASE), r'\1-\2'),
]
def apply_liaisons(text):
    for p, r in LIAISONS:
        text = p.sub(r, text)
    return text

def sanitize(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('’', "'").replace('‘', "'")
    text = text.replace('—', ', ').replace('–', ', ')
    text = text.replace('«', '').replace('»', '')
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    return text.strip()

def split_sentences(text, max_chars=200):
    """Découpe en phrases courtes pour F5-TTS (qui marche mieux sur des bouts < 200 chars)."""
    text = sanitize(text)
    text = apply_liaisons(text)
    parts = re.split(r'(?<=[.!?])\s+', text)
    out = []
    for p in parts:
        if len(p) <= max_chars:
            out.append(p)
        else:
            sub = re.split(r'(?<=[,;:])\s+', p)
            buf = ''
            for s in sub:
                if len(buf) + len(s) + 1 > max_chars and buf:
                    out.append(buf.strip())
                    buf = s
                else:
                    buf = (buf + ' ' + s).strip() if buf else s
            if buf: out.append(buf)
    return [s for s in out if s.strip()]

# === TEST PRÉALABLE : 1 phrase pour vérifier que F5 marche ===
print('\n🧪 Test de fumée sur 1 phrase…')
test_wav, test_sr, _ = f5.infer(
    ref_file=REF_AUDIO, ref_text=REF_TEXT,
    gen_text="Bonjour, ceci est un test de synthèse vocale française.",
    remove_silence=True,
)
sf.write('test_smoke.wav', test_wav, test_sr)
print(f'✓ Test OK : test_smoke.wav généré ({len(test_wav)/test_sr:.1f}s @ {test_sr} Hz)')
from IPython.display import Audio
display(Audio('test_smoke.wav'))

# === BOUCLE PRINCIPALE ===
os.makedirs('out_chapters', exist_ok=True)
chapter_wavs = []
TOTAL_TIME = time.time()
fail_count = 0
MAX_FAILS_PER_CHAPTER = 5  # si plus de 5 échecs d'affilée → on stoppe

for ci, ch in enumerate(selected_chapters):
    print(f'\n📖 Ch.{ci+1}/{len(selected_chapters)} : {ch["title"]}')
    sentences = split_sentences(ch['text'])
    print(f'   {len(sentences)} phrases')
    audio_segments = []
    sr_out = 24000
    consec_fails = 0
    for sent in tqdm(sentences, desc=f'Ch.{ci+1}'):
        try:
            wav, sr, _ = f5.infer(
                ref_file=REF_AUDIO, ref_text=REF_TEXT,
                gen_text=sent,
                remove_silence=True,
            )
            audio_segments.append(wav)
            sr_out = sr
            consec_fails = 0
        except Exception as e:
            fail_count += 1
            consec_fails += 1
            print(f'   ⚠ phrase échouée : {type(e).__name__}: {e}')
            print(f'     texte : {sent[:100]!r}')
            if consec_fails >= MAX_FAILS_PER_CHAPTER:
                raise RuntimeError(
                    f'Trop d\'échecs consécutifs ({consec_fails}) au chapitre {ci+1}. '
                    f'Stop pour debug. Voir l\'erreur ci-dessus.'
                )
    if not audio_segments:
        print(f'   ⚠ aucun audio généré pour ce chapitre, on saute')
        continue
    pause = np.zeros(int(sr_out * 0.3), dtype=np.float32)
    full = np.concatenate([np.concatenate([s, pause]) for s in audio_segments])
    out_path = f'out_chapters/ch{ci+1:02d}.wav'
    sf.write(out_path, full, sr_out)
    chapter_wavs.append({'title': ch['title'], 'path': out_path, 'duration': len(full)/sr_out})
    print(f'   ✓ {out_path}  ({len(full)/sr_out/60:.1f} min)')

print(f'\n=== Total : {(time.time()-TOTAL_TIME)/60:.1f} min de génération, {fail_count} phrases échouées ===')


## 9️⃣ Assemblage M4B avec chapitres + téléchargement

In [ ]:
import subprocess

# Génère la liste ffmpeg + metadata chapitres
concat_list = 'concat.txt'
meta_file = 'chapters.txt'

with open(concat_list, 'w', encoding='utf-8') as f:
    for cw in chapter_wavs:
        f.write(f"file '{cw['path']}'\n")

# Metadata FFmpeg avec chapter markers
with open(meta_file, 'w', encoding='utf-8') as f:
    f.write(';FFMETADATA1\n')
    start_ms = 0
    for cw in chapter_wavs:
        end_ms = start_ms + int(cw['duration'] * 1000)
        f.write(f'[CHAPTER]\nTIMEBASE=1/1000\nSTART={start_ms}\nEND={end_ms}\ntitle={cw["title"]}\n')
        start_ms = end_ms

out_m4b = 'audiobook.m4b'
print('Encodage M4B AAC + chapitres…')
subprocess.run([
    'ffmpeg', '-y',
    '-f', 'concat', '-safe', '0', '-i', concat_list,
    '-i', meta_file, '-map_metadata', '1',
    '-c:a', 'aac', '-b:a', '96k', '-ac', '1',
    '-f', 'mp4',
    out_m4b
], check=True, capture_output=True)
sz = os.path.getsize(out_m4b)/1024/1024
print(f'✓ {out_m4b}  ({sz:.1f} Mo)')

# Téléchargement
from google.colab import files
files.download(out_m4b)

## ✅ Terminé !

Ton M4B s'est téléchargé. Tu peux le charger dans iOS Books, AntennaPod, ou tout lecteur M4B.

## 💡 Conseils

- **Re-lance la cellule 8 si crash** : F5-TTS peut rarement crasher sur une phrase. Les chapitres déjà générés sont conservés.
- **Voix personnalisée** : pour cloner ta propre voix ou un narrateur que tu aimes, décommente la cellule 4 (Option B) et upload un WAV de 5-15s + sa transcription exacte.
- **Quota Colab gratuit** : 12h de GPU/jour. Un livre = ~1h, donc tu peux en faire ~10 par jour.